# 34. ViT를 Segmentation에 그대로 쓰기 어려운 이유

이 노트북은 `33_SegFormer가_등장한_배경.ipynb` 다음 단계로, 이미지 분류용 ViT를 semantic segmentation에 바로 적용할 때 생기는 문제를 정리합니다.

ViT는 patch token을 Transformer encoder에 넣고 class token으로 이미지를 분류합니다. 하지만 segmentation은 이미지 전체 class가 아니라 각 위치의 class map을 예측해야 하므로 구조적 요구가 다릅니다.

이번 노트북의 목표는 다음과 같습니다.

- class token 중심 분류와 dense prediction의 차이를 이해합니다.
- patch size가 segmentation 해상도에 주는 영향을 확인합니다.
- 단일 scale feature의 한계를 이해합니다.
- SegFormer가 왜 계층적 encoder와 decoder를 사용하는지 연결합니다.

In [ ]:
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.unicode_minus'] = False

## 34-1. Class token은 전체 이미지 요약에 가깝다

ViT 분류 모델의 최종 출력은 보통 class token을 사용합니다. 이는 이미지 전체를 대표하는 벡터이므로, 픽셀마다 class를 예측해야 하는 segmentation과는 출력 형태가 맞지 않습니다.

```text
classification: [CLS] -> class score
segmentation: every location -> class score
```

In [ ]:
labels = ['CLS'] + [f'P{i}' for i in range(1, 17)]
fig, ax = plt.subplots(figsize=(10, 2.5))
ax.axis('off')
ax.set_title('ViT classification token sequence')
for i, label in enumerate(labels):
    color = '#fee2e2' if label == 'CLS' else '#dbeafe'
    edge = '#dc2626' if label == 'CLS' else '#2563eb'
    ax.add_patch(plt.Rectangle((i, 0.4), 0.75, 0.45, facecolor=color, edgecolor=edge))
    ax.text(i + 0.375, 0.625, label, ha='center', va='center', fontsize=8)
ax.annotate('classification head는 주로 CLS를 사용', xy=(0.4, 0.9), xytext=(1.8, 1.35), arrowprops=dict(arrowstyle='->'))
ax.set_xlim(-0.2, 17.2)
ax.set_ylim(0.2, 1.6)
plt.show()

## 34-2. Patch size와 출력 해상도

`224 x 224` 이미지를 `16 x 16` patch로 나누면 token grid는 `14 x 14`입니다. 이 grid에서 바로 segmentation map을 만들면 매우 거친 결과가 됩니다.

In [ ]:
h, w = 224, 224
patch_sizes = [8, 16, 32]
for p in patch_sizes:
    print(f'patch {p}x{p}: token grid = {h // p} x {w // p}, tokens = {(h // p) * (w // p)}')

In [ ]:
fine_mask = np.zeros((64, 64))
yy, xx = np.ogrid[:64, :64]
fine_mask[(xx - 32) ** 2 + (yy - 32) ** 2 < 18 ** 2] = 1
coarse = fine_mask.reshape(8, 8, 8, 8).mean(axis=(1, 3)) > 0.5
upsampled = np.repeat(np.repeat(coarse, 8, axis=0), 8, axis=1)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, data, title in zip(axes, [fine_mask, coarse, upsampled], ['원래 mask', 'patch-level 예측', 'nearest upsample']):
    ax.imshow(data, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 34-3. 단일 scale feature의 한계

Segmentation은 작은 물체와 큰 물체를 동시에 다룹니다. 단일 해상도의 token grid만 사용하면 작은 물체의 경계가 사라지거나, 큰 물체의 문맥을 충분히 보지 못할 수 있습니다.

SegFormer는 encoder stage별로 서로 다른 해상도의 feature를 만들고 decoder에서 합칩니다.

In [ ]:
stages = [('stage 1', 56, 56), ('stage 2', 28, 28), ('stage 3', 14, 14), ('stage 4', 7, 7)]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.axis('off')
for i, (name, hh, ww) in enumerate(stages):
    x = i * 1.9
    size = 1.4 / (i + 1) + 0.25
    ax.add_patch(plt.Rectangle((x, 1 - size / 2), size, size, facecolor='#dcfce7', edgecolor='#16a34a', linewidth=2))
    ax.text(x + size / 2, 1, f'{hh}x{ww}', ha='center', va='center')
    ax.text(x + size / 2, 0.15, name, ha='center', va='center')
    if i < len(stages) - 1:
        ax.annotate('', xy=(x + 1.65, 1), xytext=(x + size + 0.1, 1), arrowprops=dict(arrowstyle='->'))
ax.set_xlim(-0.2, 7.2)
ax.set_ylim(-0.1, 1.8)
ax.set_title('SegFormer는 stage별 multi-scale feature를 사용')
plt.show()

## 정리

- ViT의 class token 중심 구조는 이미지 전체 분류에는 좋지만 dense prediction에는 바로 맞지 않습니다.
- patch size가 클수록 segmentation 출력은 거칠어집니다.
- segmentation에는 여러 해상도의 feature와 decoder가 필요합니다.
- SegFormer는 이 문제를 MiT encoder의 multi-scale feature와 MLP decoder로 해결합니다.

다음 노트북 `35_MiT_Mix_Transformer_Encoder_구조.ipynb`에서는 SegFormer encoder인 MiT 구조를 살펴봅니다.